In [58]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from bk_tools import BASE_DIR
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Import all necessary libraries for data manipulation, visualization and deep learning.

In [59]:
import tensorflow as tf
print("GPUs:", tf.config.list_physical_devices('GPU'))


GPUs: []


In [60]:

BASE_DIR = Path.cwd()
path_db = BASE_DIR / "BreaKHis_v1" / "histology_slides" / "breast"


def packup_details(f):
    p = Path(f)

    # only filename, cross-platform safe
    filename = p.name                          # SOB_B_A-14-22549G-100-001.png
    stem = p.stem                             # SOB_B_A-14-22549G-100-001

    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected filename format: {filename}")

    example = parts[0]                        # SOB
    is_malign = 1 if parts[1] == "M" else 0   # M -> malignant, B -> benign

    names = parts[2].split("-")
    if len(names) < 5:
        raise ValueError(f"Unexpected tumor name format: {filename}")

    class_tumor = names[0]
    year = int(names[1]) + 2000
    patient_id = names[2]
    zoom = int(names[3])
    file_id = names[4]

    return {
        "patient_id": patient_id,
        "file_id": file_id,
        "example": example,
        "class": class_tumor,
        "year": year,
        "zoom": zoom,
        "file_path": str(p),
        "is_malign": is_malign,
    }


def print_file_details(f):
    p = Path(f)
    filename = p.name
    stem = p.stem

    parts = stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected filename format: {filename}")

    print("Type of example:", parts[0])
    print("State:", parts[1], "(", 1 if parts[1] == "M" else 0, ")")

    nm = parts[2].split("-")
    if len(nm) < 5:
        raise ValueError(f"Unexpected tumor name format: {filename}")

    print("Class:", nm[0])
    print("Year:", nm[1])
    print("Patient ID:", nm[2])
    print("Zoom:", nm[3])
    print("File ID:", nm[4])


def prepare_data_table(rootpath=path_db) -> pd.DataFrame:
    rootpath = Path(rootpath)

    # cross-platform recursive PNG search
    files = list(rootpath.rglob("*.png"))

    if not files:
        raise FileNotFoundError(f"No PNG files found under: {rootpath}")

    datas = []
    bad_files = []

    for f in files:
        try:
            datas.append(packup_details(f))
        except Exception as e:
            bad_files.append((str(f), str(e)))

    df = pd.DataFrame(datas)

    print("DataFrame shape:", df.shape)
    print("DataFrame columns:", df.columns.tolist())
    print("Parsed files:", len(datas))
    print("Failed files:", len(bad_files))

    if bad_files:
        print("\nFirst 10 problematic files:")
        for fp, err in bad_files[:10]:
            print(fp, "->", err)

    return df

In [61]:
# Prepare the data table using the custom function from bk_tools and display its information.
df = prepare_data_table()
df.info()
df["class"].unique()

DataFrame shape: (7909, 8)
DataFrame columns: ['patient_id', 'file_id', 'example', 'class', 'year', 'zoom', 'file_path', 'is_malign']
Parsed files: 7909
Failed files: 0
<class 'pandas.DataFrame'>
RangeIndex: 7909 entries, 0 to 7908
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   patient_id  7909 non-null   str  
 1   file_id     7909 non-null   str  
 2   example     7909 non-null   str  
 3   class       7909 non-null   str  
 4   year        7909 non-null   int64
 5   zoom        7909 non-null   int64
 6   file_path   7909 non-null   str  
 7   is_malign   7909 non-null   int64
dtypes: int64(3), str(5)
memory usage: 494.4 KB


<StringArray>
['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']
Length: 8, dtype: str

In [62]:
# Patient-wise split with class coverage in all splits (train/val/test).
chosen_zoom = 40
test_val_size = 0.4
seed = 42

df_zoom = df[df["zoom"] == chosen_zoom].copy()
print(f"Selected zoom: {chosen_zoom}")
print(f"Total samples at selected zoom: {len(df_zoom)}")
print(f"Total unique patients at selected zoom: {df_zoom['patient_id'].nunique()}")

print("\nInitial image-level class distribution:")
print(df_zoom["class"].value_counts())

# Remove patients that contain multiple classes to avoid leakage/noisy labels.
patient_class_counts = df_zoom.groupby("patient_id")["class"].nunique()
problematic_patients = patient_class_counts[patient_class_counts > 1].index.tolist()
if len(problematic_patients) > 0:
    print(f"\nRemoving {len(problematic_patients)} patient(s) with multiple classes.")
    print("Problematic patient IDs:", problematic_patients)
    df_zoom = df_zoom[~df_zoom["patient_id"].isin(problematic_patients)].copy()

patient_level = df_zoom[["patient_id", "class"]].drop_duplicates().reset_index(drop=True)
print("\nClean patient-level class distribution:")
print(patient_level["class"].value_counts())

rng = np.random.default_rng(seed)
train_patients, val_patients, test_patients = [], [], []

for cls_name, grp in patient_level.groupby("class"):
    patients = grp["patient_id"].to_numpy().copy()
    rng.shuffle(patients)
    n = len(patients)

    if n < 3:
        raise ValueError(
            f"Class {cls_name} has only {n} patient(s). Need at least 3 for train/val/test coverage."
        )

    # Allocate at least 1 patient to each split for every class.
    n_temp = int(round(n * test_val_size))
    n_temp = max(2, min(n - 1, n_temp))
    n_train = n - n_temp

    n_val = n_temp // 2
    n_test = n_temp - n_val
    if n_val == 0:
        n_val, n_test = 1, n_temp - 1
    if n_test == 0:
        n_test, n_val = 1, n_temp - 1

    train_patients.extend(patients[:n_train].tolist())
    val_patients.extend(patients[n_train:n_train + n_val].tolist())
    test_patients.extend(patients[n_train + n_val:n_train + n_val + n_test].tolist())

train_df = df_zoom[df_zoom["patient_id"].isin(train_patients)].reset_index(drop=True)
val_df = df_zoom[df_zoom["patient_id"].isin(val_patients)].reset_index(drop=True)
test_df = df_zoom[df_zoom["patient_id"].isin(test_patients)].reset_index(drop=True)

# Safety checks: no patient overlap.
assert set(train_df["patient_id"]) & set(val_df["patient_id"]) == set()
assert set(train_df["patient_id"]) & set(test_df["patient_id"]) == set()
assert set(val_df["patient_id"]) & set(test_df["patient_id"]) == set()

print("\nPatient split check:")
print("Train patients:", train_df["patient_id"].nunique())
print("Val patients  :", val_df["patient_id"].nunique())
print("Test patients :", test_df["patient_id"].nunique())

print("\nFinal image-level class distribution:")
print("\nTrain:")
print(train_df["class"].value_counts())
print("\nValidation:")
print(val_df["class"].value_counts())
print("\nTest:")
print(test_df["class"].value_counts())

print("\nFinal sample counts:")
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

print("\nValidation class counts:")
print(val_df["class"].value_counts())

Selected zoom: 40
Total samples at selected zoom: 1995
Total unique patients at selected zoom: 81

Initial image-level class distribution:
class
DC    864
F     253
MC    205
LC    156
TA    149
PC    145
A     114
PT    109
Name: count, dtype: int64

Removing 1 patient(s) with multiple classes.
Problematic patient IDs: ['13412']

Clean patient-level class distribution:
class
DC    37
F     10
MC     9
TA     7
PC     6
LC     4
A      4
PT     3
Name: count, dtype: int64

Patient split check:
Train patients: 46
Val patients  : 16
Test patients : 18

Final image-level class distribution:

Train:
class
DC    506
F     173
MC    102
TA    101
PC     98
LC     73
A      50
PT     38
Name: count, dtype: int64

Validation:
class
DC    134
PT     58
F      48
A      35
MC     31
LC     31
PC     30
TA     16
Name: count, dtype: int64

Test:
class
DC    192
MC     72
TA     32
F      32
A      29
LC     20
PC     17
PT     13
Name: count, dtype: int64

Final sample counts:
Train: 1141
Validat

In [63]:
CLASS_NAMES = ['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(CLASS_NAMES)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

train_df["label"] = train_df["class"].map(class_to_idx)
val_df["label"] = val_df["class"].map(class_to_idx)
test_df["label"] = test_df["class"].map(class_to_idx)

In [64]:
# ESKİ HALİ: IMG_SIZE = (230, 230)
# YENİ HALİ:
IMG_SIZE = (224, 224) 
BATCH_SIZE = 32
NUM_CLASSES = 8
AUTOTUNE = tf.data.AUTOTUNE

# Fonksiyonun güncellenmiş (temizlenmiş) hali:
def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE) # Artık (224, 224) yapacak
    image = tf.cast(image, tf.float32)
    # tf.image.normalize ve preprocess_input sildik (DenseNet'in kendi Lambda katmanı hallediyor)
    return image, label

    
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2), # 0.1'den 0.2'ye çıkarılabilir
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    tf.keras.layers.RandomContrast(0.1), # <--- YENİ EKLENDİ
    tf.keras.layers.RandomBrightness(0.1) # <--- YENİ EKLENDİ
])


In [65]:
def make_dataset(df, training=False):
    image_paths = df["file_path"].values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

    ds = ds.map(load_and_preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()

    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

In [66]:
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

In [67]:
from tensorflow.keras import layers, models
from tensorflow.keras import applications
from tensorflow.keras.applications import densenet # Bunu import ettiğinden emin ol
def build_pretrained_sequential_model(input_shape=(224, 224, 3), num_classes=8):

    base_model = tf.keras.applications.DenseNet121(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape,
        classes=8,
        classifier_activation='softmax'
    )


    model = models.Sequential([
        tf.keras.Input(shape=input_shape),
        layers.Lambda(lambda x: densenet.preprocess_input(x)),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.6),  # <--- BURAYI AKTİF EDİN VE 0.6 YAPIN
        layers.Dense(
            num_classes, 
            activation="softmax", 
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        )
    ])


    return model, base_model

In [68]:
# Class weights from provided train class counts.
class_counts = {
    "DC": 449,
    "F": 158,
    "MC": 89,
    "PC": 88,
    "TA": 83,
    "LC": 58,
    "A": 47,
    "PT": 39,
}

total_samples = sum(class_counts.values())
num_classes = len(class_counts)

raw_class_weight_by_name = {
    cls_name: total_samples / (num_classes * count)
    for cls_name, count in class_counts.items()
}

min_w = min(raw_class_weight_by_name.values())
class_weight_by_name = {
    cls_name: weight / min_w
    for cls_name, weight in raw_class_weight_by_name.items()
}

class_weight_map = {
    class_to_idx[cls_name]: weight
    for cls_name, weight in class_weight_by_name.items()
}

raw = {c: total_samples / (num_classes * n) for c, n in class_counts.items()}
soft = {c: (w ** 0.5) for c, w in raw.items()}   # soften
m = min(soft.values())
soft = {c: min(max(w / m, 1.0), 4.0) for c, w in soft.items()}  # min=1, max=4
class_weight_map = {class_to_idx[c]: w for c, w in soft.items()}

print("Raw class weight by class name:")
for cls_name in CLASS_NAMES:
    print(f"{cls_name}: {raw_class_weight_by_name[cls_name]:.4f}")

print("\nRescaled class weight by class name (min=1.0):")
for cls_name in CLASS_NAMES:
    print(f"{cls_name}: {class_weight_by_name[cls_name]:.4f}")

print("\nClass weight map (label -> weight):")
print(class_weight_map)


class_weight_map[class_to_idx["DC"]] *= 0.7

def get_advanced_metrics():
    return [
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc"),
        tf.keras.metrics.SparseCategoricalCrossentropy(name="cross_entropy"),
    ]

Raw class weight by class name:
MC: 1.4199
PC: 1.4361
DC: 0.2815
LC: 2.1789
A: 2.6888
TA: 1.5226
F: 0.7998
PT: 3.2404

Rescaled class weight by class name (min=1.0):
MC: 5.0449
PC: 5.1023
DC: 1.0000
LC: 7.7414
A: 9.5532
TA: 5.4096
F: 2.8418
PT: 11.5128

Class weight map (label -> weight):
{2: 1.0, 6: 1.6857556619803282, 0: 2.246095238458227, 1: 2.2588210923560825, 5: 2.32586296978495, 3: 2.782333429038444, 4: 3.090823755790954, 7: 3.3930547465109537}


In [69]:
from bk_tools import BASE_DIR

model, base_model = build_pretrained_sequential_model(
    input_shape=(224, 224, 3),
    num_classes=NUM_CLASSES
)

# 1. TÜM GÖVDEYİ EĞİTİME AÇIYORUZ (TEK AŞAMA)
base_model.trainable = True

# 3. DERLEME: Ağırlıkları şoka uğratmamak için düşük bir Learning Rate (5e-5) kullanıyoruz.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2_acc")
    ]
)

MODEL_DIR = BASE_DIR/"models"
MODEL_DIR.mkdir(exist_ok=True)

# Callbacks aynen kalıyor
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10, 
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/best_model_densenet.keras",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    )
]

model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_4 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │         8,200 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,045,704 (26.88 MB)

 Trainable params: 6,962,056 (26.56 MB)

 Non-trainable params: 83,648 (326.75 KB)

In [70]:
# Tek seferde uçtan uca (End-to-End) eğitim
history_single_stage = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60, # EarlyStopping zaten en iyi yerde keseceği için bol veriyoruz
    class_weight=class_weight_map,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/60


W0000 00:00:1777312863.370282 1319271 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


10/36 ━━━━━━━━━━━━━━━━━━━━ 1:09 3s/step - accuracy: 0.2198 - loss: 5.0883 - top2_acc: 0.3941

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    history_dict = history.history

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["loss"], label="train_loss")
    plt.plot(history_dict["val_loss"], label="val_loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(history_dict["accuracy"], label="train_accuracy")
    plt.plot(history_dict["val_accuracy"], label="val_accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    if "top2_acc" in history_dict and "val_top2_acc" in history_dict:
        plt.figure(figsize=(12, 5))
        plt.plot(history_dict["top2_acc"], label="train_top2_acc")
        plt.plot(history_dict["val_top2_acc"], label="val_top2_acc")
        plt.title("Top-2 Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Top-2 Accuracy")
        plt.legend()
        plt.show()

plot_training_history(history_single_stage)

In [ ]:
test_loss ,test_acc, cross_ent = model.evaluate(test_ds, verbose=1)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")



y_true = test_df["label"].values
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

In [ ]:
from sklearn.metrics import classification_report

CLASS_NAMES = ['MC', 'PC', 'DC', 'LC', 'A', 'TA', 'F', 'PT']

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(cm, class_names, title="Confusion Matrix", normalize=False, cmap="Blues"):
    cm = np.asarray(cm)

    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1, row_sums)
        cm_to_plot = cm.astype(np.float64) / row_sums

        annot = np.empty_like(cm, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                annot[i, j] = f"{cm[i, j]}\n({cm_to_plot[i, j] * 100:.1f}%)"
        fmt = ""
    else:
        cm_to_plot = cm
        annot = cm
        fmt = "d"

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm_to_plot,
        annot=annot,
        fmt=fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True,
        linewidths=0.5,
        linecolor="white",
        square=True
    )
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, CLASS_NAMES, title="Confusion Matrix (Counts)", normalize=False)
plot_confusion_matrix(cm, CLASS_NAMES, title="Confusion Matrix (Row-Normalized)", normalize=True)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import densenet

# 1. Eğitim setinden rastgele bir görselin yolunu alalım
sample_path = train_df["file_path"].iloc[0]
raw_img = tf.io.read_file(sample_path)
raw_img = tf.image.decode_png(raw_img, channels=3)
raw_img_resized = tf.image.resize(raw_img, (224, 224)) # Model 224 bekliyor

print("--- 1. HAM GÖRSEL (Beklenen: 0 ile 255 arası) ---")
print(f"Min: {tf.reduce_min(raw_img_resized):.2f}, Max: {tf.reduce_max(raw_img_resized):.2f}")

# 2. DOĞRU SENARYO (Sadece DenseNet Preprocess)
# DenseNet'e sadece 0-255 arası float32 veri gitmelidir.
correct_input = tf.cast(raw_img_resized, tf.float32)
correct_processed = densenet.preprocess_input(correct_input)

print("\n--- 2. DOĞRU DENSENET GİRDİSİ (Ağın Görmesi Gereken) ---")
print("Değerler ImageNet standartlarında olmalı (örn: ~ -2.0 ile +2.5 arası)")
print(f"Min: {tf.reduce_min(correct_processed):.4f}, Max: {tf.reduce_max(correct_processed):.4f}, Mean: {tf.reduce_mean(correct_processed):.4f}")

# 3. SENİN SENARYON (Double Preprocessing Faciası)
# ConvNeXt preprocessing ve üstüne DenseNet preprocessing yapıldığında ne oluyor?
from tensorflow.keras.applications.convnext import preprocess_input as conv_prep

buggy_input = tf.cast(raw_img_resized, tf.float32)
buggy_input = conv_prep(buggy_input) # Senin load_and_preprocess_image'deki işlem
buggy_double_processed = densenet.preprocess_input(buggy_input) # Senin Sequential modelindeki Lambda işlemi

print("\n--- 3. ŞU ANKİ MODELİNE GİDEN GİRDİ (Körleşen Ağ) ---")
print("Değerlerin ne kadar sıfıra ezildiğine dikkat et!")
print(f"Min: {tf.reduce_min(buggy_double_processed):.8f}, Max: {tf.reduce_max(buggy_double_processed):.8f}")